In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import sys
import os
import time
from tqdm import tqdm

# The code is desigend to be run on both CPU and GPU. 
# If cupy is available, it will use GPU for computation, 
# otherwise it will fall back to numpy on CPU.
if torch.cuda.is_available(): 
    dev = "cuda:0" 
    import cupy as cp 
else: 
    dev = "cpu" 
    import numpy as cp  
device = torch.device(dev) 

# These directories are used to store the custom modules, data and results. You can change them to your own directories.
code_dir = '/n/data2/hms/neurobio/harvey/siyan/arctic/'
data_dir = f'/n/data2/hms/neurobio/harvey/siyan/data/mesoscope/'
root_dir = f'/n/data2/hms/neurobio/harvey/siyan/result/result1122/'

# Load custom modules
os.sys.path.insert(0,code_dir)
from scripts.Ymaze_simulation.Environment import Ymaze_obstacle
from scripts.Ymaze_simulation.LoadData import load_charlotte_delay
from scripts.Ymaze_RL.Ckpt_utils import save_ckpt,load_ckpt
from scripts.Ymaze_RL.RL_TrainEval_torch import evaluation_loop,training_loop_batch
from scripts.Ymaze_RL.RL_Agent_torch import RNN, Actor, Critic
from scripts.Ymaze_analyses.Utils import model_correctness_RL

# Load example data session, and the weights of the data-derived agent trained by ARCTIC

In [2]:
filename='CA63/190703/'
cv_fold = 0

delay, binary_labels, frame_trial, activity, behavior=load_charlotte_delay(data_dir,filename)
n_neuron = activity.shape[0]
n_frame = activity.shape[1]
n_trial = binary_labels.shape[1]
cor_trial_idxes=np.where(binary_labels[2]==1)[0]
n_cor_trials=cor_trial_idxes.shape[0]
cues=np.array(binary_labels[0])
choices=np.array(binary_labels[1][cor_trial_idxes])
activity[activity>=1]=1-1e-4
behavior[np.isnan(behavior)] = 0
frame_trial = np.array(frame_trial)

result_dir=root_dir+filename+f'cv{cv_fold}/'
J_neu=cp.load(result_dir+'Jneu_30.npy')
J_beh=np.load(result_dir+'Jbeh_30.npy')
train_trial_idxes = np.load(result_dir+ 'train_trial_idxes.npy')
train_trials=np.array([cor_trial_idxes[i] in train_trial_idxes for i in range(cor_trial_idxes.shape[0])])
test_trials=~train_trials

# Set up the torch objects, and define the RL curriculum stages

In [3]:
R0=2
R1=R2=1
R3=-1
lr_actor=1e-3
lr_critic=3e-2
weight_decay=1e-2
rewards=np.array([R0,R1,R2,R3])
#rewards are normalized so that mean absoluate value is 1
rewards=rewards/np.mean(np.abs(rewards))
param=np.hstack((rewards,np.array([lr_actor,lr_critic,weight_decay])))

rnn=RNN(dtData=0.186, dt=0.0093, tau=0.1, N=n_neuron,phi='modifiedtanh',J_neu=J_neu,cur_noise=0)
actor = Actor(N=n_neuron,J_beh=J_beh).to(device)
critic = Critic().to(device)
actor_optimizer = optim.Adam(actor.parameters(), lr=param[4],weight_decay=param[6])
critic_optimizer = optim.Adam(critic.parameters(), lr=param[5],weight_decay=param[6])

# Define the curriculum of increased obstacle size. 
# each obstacle is defined as [posF,posL_left,posL_right], where the obstacle is located at 
# forward position posF, and occupies the lateral space between posL_left and posL_right
# Here negative value for posL is leftward, all units in cm)
obstacles_1=[[180,-4,7],[180,-8,7],[180,-12,7],[180,-16,7]]
obstacles_2=[[230,-35,-30],[230,-35,-25],[230,-35,-20],[230,-35,-15]]

result_dir_RL=result_dir + f'RL/'
if not os.path.exists(result_dir_RL):
    os.makedirs(result_dir_RL)

from dataclasses import dataclass
@dataclass
class stage_config:
    idx: int
    max_epochs: int
    threshold: float
    epochs_trained: int = None           
    best_epoch: int = None     
    best_metric: float = None  
    met_threshold: bool = None    

stage0 = stage_config(idx=0,max_epochs=30,threshold=0.85)
stage1 = stage_config(idx=1,max_epochs=30,threshold=0.85)
stage2 = stage_config(idx=2,max_epochs=30,threshold=0.85)
stage3 = stage_config(idx=3,max_epochs=30,threshold=0.85)

# Train and evaluation

In [ ]:
learning_curve=[]
stages=[stage0,stage1,stage2,stage3]

# Iterate through curriculum stages, promote to next stage if 
# training performance meets the threshold or exceeding max epochs
for stage in stages:
    best_metric = float("-inf")
    best_epoch = 0
    epochs_trained = 0
    met_threshold=False
    # Implent curriculum rollback, if performance didn't increase for certain number of epochs, 
    # rollback to previous checkpoint and continue training. 
    count_rollback=0
    count_bad=0
    count_flat=0
    bad_threshold=0.6
    appendix=''
    t0=time.perf_counter()
    for epoch in tqdm(range(stage.max_epochs)):
        env=Ymaze_obstacle(delay,[obstacles_1[stage.idx],obstacles_2[stage.idx]])
        torch.cuda.empty_cache()

        #train on training trials, evaluate on training and testing trials and save to learning curve
        training_loop_batch(rnn, actor, critic, actor_optimizer, critic_optimizer, env, param, 
                            cues, activity, behavior,frame_trial,train_trial_idxes,1,True)
        _,beh_output_eval,frame_trial_eval,_,_,_=\
                        evaluation_loop(rnn, actor, critic, env, param, cues, activity, behavior,frame_trial,cor_trial_idxes)
        frame_trial_eval=np.array(frame_trial_eval)
        correct_m=model_correctness_RL(cp.asnumpy(beh_output_eval),frame_trial_eval,cor_trial_idxes,choices)
        train_frac=np.sum(correct_m[train_trials])/np.sum(train_trials)
        test_frac=np.sum(correct_m[test_trials])/np.sum(test_trials)
        train_frac_left=np.sum(correct_m[np.logical_and(train_trials,choices==0)])/\
            np.sum(np.logical_and(train_trials,choices==0))
        test_frac_left=np.sum(correct_m[np.logical_and(test_trials,choices==0)])/\
            np.sum(np.logical_and(test_trials,choices==0))
        learning_curve.append(np.array([f'{stage.idx}{appendix}',train_frac,test_frac,train_frac_left,test_frac_left]))
        np.save(result_dir_RL+'learning_curve.npy',np.array(learning_curve))
        epochs_trained+=1
        
        is_better = train_frac >= best_metric
        if is_better:
            best_metric = train_frac
            best_epoch = epoch
            #save checkpoint
            np.save(result_dir_RL+f'beh_output_stage{stage.idx}_epoch{epoch}.npy',beh_output_eval)
            np.save(result_dir_RL+f'frame_trial_stage{stage.idx}_epoch{epoch}.npy',frame_trial_eval)
            save_ckpt(result_dir_RL+f"ckpts/actor_stage{stage.idx}_epoch{epoch}.pt",actor,actor_optimizer)
            save_ckpt(result_dir_RL+f"ckpts/critic_stage{stage.idx}_epoch{epoch}.pt",critic,critic_optimizer)
            count_flat=0
        else:
            count_flat+=1
           
        met_threshold = train_frac >= stage.threshold
        if met_threshold:
            break
            
        #rollback procedure
        is_bad = train_frac < bad_threshold
        if is_bad:
            count_bad+=1
        else:
            count_bad=0
            
        if count_bad == 4 or count_flat == 10:
            count_rollback+=1
            count_bad=0
            count_flat=0
            if count_rollback>=1:
                appendix=f'_{count_rollback}'
            if best_metric>=0.75:
                load_ckpt(result_dir_RL+f"ckpts/actor_stage{stage.idx}_epoch{best_epoch}.pt",actor,actor_optimizer)
                load_ckpt(result_dir_RL+f"ckpts/critic_stage{stage.idx}_epoch{best_epoch}.pt",critic,critic_optimizer)
            else:
                if stage.idx>=1:
                    load_ckpt(result_dir_RL+f"ckpts/actor_stage{stage.idx-1}_epoch{stages[stage.idx-1].best_epoch}.pt",actor,actor_optimizer)
                    load_ckpt(result_dir_RL+f"ckpts/critic_stage{stage.idx-1}_epoch{stages[stage.idx-1].best_epoch}.pt",critic,critic_optimizer)
                else:
                    actor = Actor(N=n_neuron,J_beh=J_beh).to(device)
                    critic = Critic().to(device)
                    actor_optimizer = optim.Adam(actor.parameters(), lr=param[4],weight_decay=param[6])
                    critic_optimizer = optim.Adam(critic.parameters(), lr=param[5],weight_decay=param[6])

    elapsed = time.perf_counter() - t0
    print(f"Stage {stage.idx}: {epoch} epochs in {elapsed:.2f} s")

    #after each stage load checkpoint for this stage
    stage.epochs_trained=epochs_trained
    stage.best_epoch=best_epoch
    stage.best_metric=best_metric
    stage.met_threshold=met_threshold
    load_ckpt(result_dir_RL+f"ckpts/actor_stage{stage.idx}_epoch{best_epoch}.pt",actor,actor_optimizer)
    load_ckpt(result_dir_RL+f"ckpts/critic_stage{stage.idx}_epoch{best_epoch}.pt",critic,critic_optimizer)

  0%|          | 0/30 [00:00<?, ?it/s]

  3%|▎         | 1/30 [1:43:28<50:00:51, 6208.69s/it]